# BENCHMARK: Particionamento vs Z-Order vs Liquid Clustering
**Databricks DBR 13.3+ recomendado**

**Autor: Marcela Monteiro Montenegro Gallo**

> Roda cada célula em sequência. A célula 6 gera a tabela de resultados para screenshot.

## Célula 1  Configuração

In [ ]:
CATALOG    = "main"
SCHEMA     = "benchmark_demo"
N_ROWS     = 50_000_000
N_CLIENTES = 500_000

spark.conf.set("spark.sql.shuffle.partitions", "200")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")

print(f"Gerando {N_ROWS:,} linhas com {N_CLIENTES:,} clientes distintos")
print(f"Catalog: {CATALOG}.{SCHEMA}")

## Célula 2  Gera massa de dados

In [ ]:
from pyspark.sql import functions as F
import time

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

df_raw = (
    spark.range(N_ROWS)
    .withColumn("venda_id",   F.col("id"))
    .withColumn("cliente_id", (F.rand() * N_CLIENTES).cast("long"))
    .withColumn("produto_id", (F.rand() * 10_000).cast("long"))
    .withColumn("regiao",     F.element_at(
        F.array(F.lit("sudeste"), F.lit("sul"), F.lit("nordeste"),
                F.lit("norte"),   F.lit("centro-oeste")),
        (F.rand() * 5 + 1).cast("int")
    ))
    .withColumn("data_venda", F.date_add(F.lit("2022-01-01"), (F.rand() * 730).cast("int")))
    .withColumn("valor",      (F.rand() * 5000 + 10).cast("decimal(18,2)"))
    .withColumn("status",     F.element_at(
        F.array(F.lit("aprovado"), F.lit("cancelado"), F.lit("pendente")),
        (F.rand() * 3 + 1).cast("int")
    ))
    .withColumn("canal",      F.element_at(
        F.array(F.lit("app"), F.lit("web"), F.lit("loja"), F.lit("telefone")),
        (F.rand() * 4 + 1).cast("int")
    ))
    .drop("id")
)

df_raw.cache()
count = df_raw.count()
print(f" {count:,} linhas geradas")

## Célula 3  Cria as 4 versões da tabela

In [ ]:
BASE = f"{CATALOG}.{SCHEMA}"

print("Criando tabela 1/4: sem_otimizacao...")
spark.sql(f"DROP TABLE IF EXISTS {BASE}.vendas_sem_otimizacao")
df_raw.write.format("delta").mode("overwrite").saveAsTable(f"{BASE}.vendas_sem_otimizacao")
print("   vendas_sem_otimizacao criada")

print("Criando tabela 2/4: particionada_data...")
spark.sql(f"DROP TABLE IF EXISTS {BASE}.vendas_particionada_data")
df_raw.write.format("delta").mode("overwrite").partitionBy("data_venda").saveAsTable(f"{BASE}.vendas_particionada_data")
print("   vendas_particionada_data criada")

print("Criando tabela 3/4: zorder_cliente...")
spark.sql(f"DROP TABLE IF EXISTS {BASE}.vendas_zorder_cliente")
df_raw.write.format("delta").mode("overwrite").saveAsTable(f"{BASE}.vendas_zorder_cliente")
spark.sql(f"OPTIMIZE {BASE}.vendas_zorder_cliente ZORDER BY (cliente_id)")
print("   vendas_zorder_cliente criada + ZORDER aplicado")

print("Criando tabela 4/4: liquid_clustering...")
spark.sql(f"DROP TABLE IF EXISTS {BASE}.vendas_liquid_clustering")
spark.sql(f"""
    CREATE TABLE {BASE}.vendas_liquid_clustering
    CLUSTER BY (cliente_id, data_venda)
    AS SELECT * FROM {BASE}.vendas_sem_otimizacao
""")
spark.sql(f"OPTIMIZE {BASE}.vendas_liquid_clustering")
print("   vendas_liquid_clustering criada + OPTIMIZE aplicado")

print("\n Todas as 4 tabelas prontas!")

## Célula 4  Função de benchmark

In [ ]:
import re

def run_benchmark(table_name, query_sql, label):
    spark.catalog.clearCache()
    spark.sql("CLEAR CACHE")
    start   = time.time()
    count   = spark.sql(query_sql).count()
    elapsed = time.time() - start
    mins    = int(elapsed // 60)
    secs    = elapsed % 60
    try:
        detail  = spark.sql(f"DESCRIBE DETAIL {table_name}").first()
        n_files = detail["numFiles"]
    except:
        n_files = "N/A"
    print(f"{''*52}")
    print(f"  Estratégia : {label}")
    print(f"  Linhas     : {count:,}")
    print(f"  Tempo      : {mins}m {secs:.1f}s")
    print(f"  Arquivos   : {n_files}")
    print(f"{''*52}")
    return {"label": label, "tempo_s": elapsed, "mins": mins, "secs": secs, "files": n_files}

print(" Função run_benchmark definida")

## Célula 5  Executa o benchmark nas 4 tabelas

In [ ]:
BASE = f"{CATALOG}.{SCHEMA}"

sample_cliente = spark.sql(f"""
    SELECT cliente_id FROM {BASE}.vendas_sem_otimizacao
    GROUP BY cliente_id ORDER BY COUNT(*) DESC LIMIT 1
""").first()["cliente_id"]

print(f" Testando com cliente_id = {sample_cliente}\n")

Q = "SELECT cliente_id, COUNT(*) AS total_vendas, SUM(valor) AS total_valor, AVG(valor) AS ticket_medio, MIN(data_venda) AS primeira_compra, MAX(data_venda) AS ultima_compra FROM {t} WHERE cliente_id = {c} GROUP BY cliente_id"

resultados = []
for tbl, label in [
    ("vendas_sem_otimizacao",    "Sem otimizacao"),
    ("vendas_particionada_data", "Particionado por data"),
    ("vendas_zorder_cliente",    "Z-Order por cliente_id"),
    ("vendas_liquid_clustering", "Liquid Clustering"),
]:
    r = run_benchmark(f"{BASE}.{tbl}", Q.format(t=f"{BASE}.{tbl}", c=sample_cliente), label)
    resultados.append(r)

## Célula 6  Resultado final para screenshot 

In [ ]:
base_tempo = resultados[0]["tempo_s"]

print("\n")
print("")
print("      BENCHMARK  Particionamento no Databricks              ")
print("      Filtro: cliente_id (alta cardinalidade)                ")
print("")
print(f"  {'Estrategia':<26} {'Tempo':>8}  {'Arquivos':>8}  {'Melhoria':>8}  ")
print("")
for r in resultados:
    melhoria  = f"{base_tempo / r['tempo_s']:.1f}x"
    tempo_fmt = f"{r['mins']}m {r['secs']:.0f}s"
    print(f"  {r['label']:<26} {tempo_fmt:>8}  {str(r['files']):>8}  {melhoria:>8}  ")
print("")

reducao = (1 - resultados[3]["tempo_s"] / base_tempo) * 100
print(f"\n Liquid Clustering foi {reducao:.1f}% mais rapido que sem otimizacao")

## Célula 7  Arquivos físicos por tabela 

In [ ]:
print(" Arquivos fisicos e tamanho por tabela:")
print("" * 55)
for tbl, label in [
    ("vendas_sem_otimizacao",    "Sem otimizacao"),
    ("vendas_particionada_data", "Particionado por data"),
    ("vendas_zorder_cliente",    "Z-Order"),
    ("vendas_liquid_clustering", "Liquid Clustering"),
]:
    d      = spark.sql(f"DESCRIBE DETAIL {BASE}.{tbl}").first()
    files  = d["numFiles"]
    size   = d["sizeInBytes"] / (1024**3)
    print(f"  {label:<26} {files:>6} arquivos  {size:.1f} GB")

## Célula 8  Queries para Spark UI (data skipping) 

Roda cada query e abre **Spark UI  SQL  último job**.
Procura por `number of files pruned` nas métricas do scan.

In [ ]:
for tbl, label in [
    ("vendas_sem_otimizacao",    "Sem otimizacao"),
    ("vendas_liquid_clustering", "Liquid Clustering"),
]:
    print(f"-- {label}")
    r = spark.sql(f"SELECT COUNT(*), SUM(valor) FROM {BASE}.{tbl} WHERE cliente_id = {sample_cliente}")
    r.show()
    print()